# 03 — Ridge and XGBoost

**Goal:** compare the tabular models with the baselines on the validation month, answer the ablation questions (does the trajectory help? does timetable slack help?), and see where and why the models gain.

1. Headline results (champion run, 100 % of training data)
2. Paired comparisons: refit, history, slack, prior
3. Breakdowns: category, current delay, disruption day
4. What the champion model uses: feature importance
5. Findings and decisions

**Setup reminder.** One model per horizon. Fit on Aug 2025 – Mar 2026, tune on April (Ridge penalty, XGBoost early stopping), refit on Aug – Apr, evaluate on May. MAE in minutes of Δd, 95 % confidence intervals from a bootstrap over whole days. `diff` = MAE(model) − MAE(reference) on the same points; negative = better.

In [1]:
# 0. Setup: predictions of the champion run, the ablation run and the baselines
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import xgboost as xgb

from swissdelay import config
from swissdelay.evaluation.metrics import evaluate, mae_diff_ci
from swissdelay.features.build import FEATURE_GROUPS
from swissdelay.models.baselines import PREDICTIONS_PATH as BASELINE_PATH
from swissdelay.viz import GRAY, SERIES, use_style

use_style()
pd.set_option("display.float_format", "{:,.3f}".format)
P = config.PROCESSED
KEYS = ["trip_key", "stop_seq", "horizon_min"]

con = duckdb.connect()
champ = con.sql(f"""
SELECT t.*, b.pred_persistence, b.pred_offset
FROM read_parquet('{P}/tabular_predictions_valid.parquet') t
JOIN read_parquet('{BASELINE_PATH}') b USING (trip_key, stop_seq, horizon_min)
""").df()
abl = pd.read_parquet(P / "tabular_predictions_valid_ablations.parquet")
for df in (champ, abl):
    df["in_common_subset"] = df["in_common_subset"].astype(bool)
    df["is_disruption_day"] = df["is_disruption_day"].astype(bool)
print(len(champ), "champion rows,", len(abl), "ablation rows")

953339 champion rows, 953339 ablation rows


## 1. Headline results

In [2]:
# 1.1 Validation, common subset: all models vs the historical baseline
MODELS = ["pred_persistence", "pred_offset", "pred_historical", "pred_ridge_full",
          "pred_xgb_norefit_full", "pred_xgb_full"]
common = champ[champ["in_common_subset"]]
headline = evaluate(common, MODELS, reference="pred_historical")
headline.pivot(index="model", columns="horizon_min", values="mae").loc[[m.removeprefix("pred_") for m in MODELS]]

horizon_min,15,30,60
model,,,
persistence,1.325,1.474,1.637
offset,1.060,1.249,1.443
historical,0.883,1.073,1.279
ridge_full,0.870,1.033,1.200
xgb_norefit_full,0.728,0.869,1.028
xgb_full,0.720,0.860,1.020


In [3]:
# 1.2 Same, with confidence intervals and the difference vs the historical baseline
headline[["horizon_min", "model", "mae", "mae_lo", "mae_hi", "p90_abs_err", "diff_vs_ref", "diff_lo", "diff_hi"]]

,horizon_min,model,mae,mae_lo,mae_hi,p90_abs_err,diff_vs_ref,diff_lo,diff_hi
0,15,persistence,1.325,1.305,1.346,2.600,0.442,0.430,0.456
1,15,offset,1.060,1.032,1.084,2.183,0.178,0.174,0.181
2,15,historical,0.883,0.855,0.905,1.883,0.000,0.000,0.000
3,15,ridge_full,0.870,0.846,0.892,1.811,-0.012,-0.016,-0.008
4,15,xgb_norefit_full,0.728,0.705,0.747,1.527,-0.155,-0.161,-0.149
5,15,xgb_full,0.720,0.698,0.740,1.515,-0.162,-0.169,-0.157
6,30,persistence,1.474,1.444,1.504,2.967,0.401,0.387,0.416
7,30,offset,1.249,1.211,1.283,2.617,0.176,0.173,0.179
8,30,historical,1.073,1.035,1.107,2.333,0.000,0.000,0.000
9,30,ridge_full,1.033,0.995,1.066,2.175,-0.040,-0.045,-0.036


In [4]:
# 1.3 Chart: MAE by horizon (95 % CI)
order = ["persistence", "offset", "historical", "ridge_full", "xgb_full"]
colors = {"persistence": GRAY, "offset": "#d4d3cd", "historical": SERIES[1], "ridge_full": SERIES[3],
          "xgb_full": SERIES[0]}
r = headline[headline["model"].isin(order)].copy()
r["plus"], r["minus"] = r["mae_hi"] - r["mae"], r["mae"] - r["mae_lo"]
fig = px.bar(r, x="horizon_min", y="mae", color="model", barmode="group", error_y="plus",
             error_y_minus="minus", category_orders={"model": order}, color_discrete_map=colors,
             title="Validation MAE by horizon (common subset)")
fig.update_layout(xaxis=dict(type="category", title="horizon (min)"), yaxis_title="MAE (min)",
                  legend_title=None)
fig.show()

> **Finding 1.1–1.3.**
> - **XGBoost (champion) reaches 0.720 / 0.860 / 1.020 min** at 15 / 30 / 60 min: −0.16 / −0.21 / −0.26 min vs the historical baseline (**−18 / −20 / −20 %**), and −46 / −42 / −38 % vs persistence. Every interval of the difference excludes 0.
> - The tail improves as much as the mean: P90 absolute error 1.52 / 1.80 / 2.11 min vs 1.88 / 2.33 / 2.78 for the historical baseline.
> - **Ridge, with the same features, barely beats the historical baseline** (−0.01 / −0.04 / −0.08 min): the useful information is there, but its effect is not linear.

In [5]:
# 1.4 Each horizon's full labelled population (not only the common subset)
full_pop = evaluate(champ, ["pred_historical", "pred_ridge_full", "pred_xgb_full"], reference="pred_historical")
full_pop.pivot(index="model", columns="horizon_min", values="mae")

horizon_min,15,30,60
model,,,
historical,0.853,1.055,1.282
ridge_full,0.834,1.017,1.203
xgb_full,0.678,0.840,1.023


> **Finding 1.4.** On each horizon's full labelled population (not only the common subset), the gain is the same: XGBoost 0.678 / 0.840 / 1.023 min vs historical 0.853 / 1.055 / 1.282 (−21 / −20 / −20 %). The common-subset restriction does not change the conclusion.

## 2. Paired comparisons

Each line compares two models **on exactly the same points**, with a day-clustered CI of the MAE difference. Ablations come from the 30 % run (same settings for every feature set).

In [6]:
# 2.1 Refit, history, slack and prior (XGBoost, validation, common subset)
def paired(df, a, b, label):
    rows = []
    for h, g in df[df["in_common_subset"]].groupby("horizon_min"):
        y, day = g["delta_min"].to_numpy(), g["operating_day"].to_numpy()
        d, lo, hi = mae_diff_ci(day, np.abs(y - g[a]), np.abs(y - g[b]))
        rows.append({"comparison": label, "horizon_min": h, "diff": d, "lo": lo, "hi": hi,
                     "significant": hi < 0 or lo > 0})
    return rows

rows = []
rows += paired(champ, "pred_xgb_full", "pred_xgb_norefit_full", "refit vs no refit (champion)")
rows += paired(abl, "pred_xgb_history_1", "pred_xgb_current", "+1 previous stop vs current")
rows += paired(abl, "pred_xgb_history_5", "pred_xgb_current", "+5 previous stops vs current")
rows += paired(abl, "pred_xgb_full", "pred_xgb_history_5", "+slack vs history_5")
rows += paired(abl, "pred_xgb_full_no_prior", "pred_xgb_full", "no prior vs full")
rows += paired(champ, "pred_xgb_full", "pred_ridge_full", "XGBoost vs Ridge (champion)")
comparisons = pd.DataFrame(rows)
comparisons.pivot(index="comparison", columns="horizon_min", values="diff").round(4)

horizon_min,15,30,60
comparison,,,
+1 previous stop vs current,-0.001,-0.000,-0.001
+5 previous stops vs current,0.002,0.002,0.003
+slack vs history_5,-0.014,-0.015,-0.011
XGBoost vs Ridge (champion),-0.150,-0.172,-0.180
no prior vs full,0.002,0.005,0.003
refit vs no refit (champion),-0.008,-0.008,-0.008


In [7]:
# 2.2 With confidence intervals
comparisons.round(4)

,comparison,horizon_min,diff,lo,hi,significant
0,refit vs no refit (champion),15,-0.008,-0.008,-0.007,True
1,refit vs no refit (champion),30,-0.008,-0.009,-0.007,True
2,refit vs no refit (champion),60,-0.008,-0.009,-0.007,True
3,+1 previous stop vs current,15,-0.001,-0.002,-0.000,True
4,+1 previous stop vs current,30,-0.000,-0.001,0.001,False
5,+1 previous stop vs current,60,-0.001,-0.002,-0.001,True
6,+5 previous stops vs current,15,0.002,0.002,0.003,True
7,+5 previous stops vs current,30,0.002,0.001,0.003,True
8,+5 previous stops vs current,60,0.003,0.002,0.004,True
9,+slack vs history_5,15,-0.014,-0.017,-0.011,True


> **Finding 2.1–2.2 (paired differences, XGBoost, minutes).**
> - **Refit** on fitting + tuning months: −0.008 at every horizon, significant. Kept.
> - **Trajectory:** +1 previous stop −0.001 (0.000 at 30 min, not significant); +5 previous stops **+0.002 to +0.003, slightly worse** (significant but negligible: extra noisy inputs). **The train's history before the current stop adds nothing** once its current delay, arrival delay and dwell are known.
> - **Timetable slack:** −0.014 / −0.015 / −0.011, significant at every horizon: the only feature group that adds information beyond the current state.
> - **Historical prior:** removing it costs +0.002 / +0.005 / +0.003 (significant but small): XGBoost rebuilds most of it from the other features.
> - **XGBoost vs Ridge** on identical features: −0.15 / −0.17 / −0.18. Non-linearity is the main source of gain.

In [8]:
# 2.3 Ablation table: MAE of every feature set (XGBoost with refit and Ridge, 30 % run)
sets = ["current", "history_1", "history_3", "history_5", "full", "full_no_prior"]
cols = [f"pred_{m}_{s}" for m in ("ridge", "xgb") for s in sets]
abl_res = evaluate(abl[abl["in_common_subset"]], ["pred_historical"] + cols, reference="pred_historical", n_boot=200)
abl_res.pivot(index="model", columns="horizon_min", values="mae").loc[
    ["historical"] + [c.removeprefix("pred_") for c in cols]]

horizon_min,15,30,60
model,,,
historical,0.883,1.073,1.279
ridge_current,0.881,1.044,1.214
ridge_history_1,0.878,1.038,1.199
ridge_history_3,0.878,1.037,1.196
ridge_history_5,0.877,1.036,1.195
ridge_full,0.869,1.031,1.193
ridge_full_no_prior,0.947,1.134,1.267
xgb_current,0.741,0.884,1.042
xgb_history_1,0.740,0.883,1.041


> **Finding 2.3.** The ablation table confirms it for both models. For XGBoost, `current` alone already gets 0.741 / 0.884 / 1.042 (most of the gain); history sets stay within ±0.002 of it; `full` (+ slack) is best. Ridge gains a little from history at 60 min (1.214 → 1.195) and **collapses without the prior** (0.947 at 15 min, worse than the historical baseline): the linear model relies on the historical median, the trees do not.

## 3. Breakdowns (validation, common subset)

In [9]:
# 3.1 By category
by_cat = evaluate(common, ["pred_historical", "pred_ridge_full", "pred_xgb_full"],
                  group_cols=("horizon_min", "category"), reference="pred_historical", n_boot=300)
by_cat.pivot_table(index="category", columns=["horizon_min", "model"], values="mae").round(3)

horizon_min         15                             30                      \
model       historical ridge_full xgb_full historical ridge_full xgb_full   
category                                                                    
EC               1.461      1.559    1.152      2.138      2.210    1.636   
IC               1.003      1.003    0.843      1.195      1.163    0.986   
IR               0.803      0.789    0.646      0.969      0.923    0.762   
RE               0.872      0.834    0.712      1.060      1.008    0.867   

horizon_min         60                      
model       historical ridge_full xgb_full  
category                                    
EC               2.444      2.500    1.874  
IC               1.429      1.346    1.165  
IR               1.161      1.071    0.910  
RE               1.257      1.192    1.022

> **Finding 3.1 (by category, XGBoost vs historical).** XGBoost improves every category: EC 1.15 / 1.64 / 1.87 min (**−21 / −23 / −23 %**, the hardest category gains most), IR 0.65 / 0.76 / 0.91 (−20 / −21 / −22 %), RE 0.71 / 0.87 / 1.02 (−18 / −18 / −19 %), IC 0.84 / 0.99 / 1.17 (−16 / −17 / −18 %). Ridge is **worse than the historical baseline for EC** at every horizon.

In [10]:
# 3.2 By current delay
common = common.assign(d0_bucket=pd.cut(common["d0_min"], [-99, 1, 3, 10, 999], right=False,
                       labels=["on time < 1", "minor 1-3", "moderate 3-10", "major > 10"]))
by_d0 = evaluate(common, ["pred_historical", "pred_ridge_full", "pred_xgb_full"],
                 group_cols=("horizon_min", "d0_bucket"), reference="pred_historical", n_boot=300)
display(by_d0.pivot_table(index="d0_bucket", columns=["horizon_min", "model"], values="mae", observed=True).round(3))
by_d0[by_d0["model"] == "xgb_full"][["horizon_min", "d0_bucket", "n", "mae", "diff_vs_ref", "diff_lo", "diff_hi"]]

horizon_min           15                             30                      \
model         historical ridge_full xgb_full historical ridge_full xgb_full   
d0_bucket                                                                     
major > 10         2.924      2.977    2.275      4.494      4.630    3.238   
minor 1-3          0.874      0.889    0.715      1.050      1.041    0.857   
moderate 3-10      1.398      1.358    0.912      2.202      1.999    1.270   
on time < 1        0.783      0.757    0.672      0.873      0.835    0.767   

horizon_min           60                      
model         historical ridge_full xgb_full  
d0_bucket                                     
major > 10         6.270      6.379    4.305  
minor 1-3          1.217      1.177    1.009  
moderate 3-10      2.867      2.372    1.487  
on time < 1        1.013      0.967    0.909

,horizon_min,d0_bucket,n,mae,diff_vs_ref,diff_lo,diff_hi
2,15,on time < 1,104633,0.672,-0.111,-0.117,-0.106
5,15,minor 1-3,61680,0.715,-0.159,-0.166,-0.153
8,15,moderate 3-10,14739,0.912,-0.486,-0.508,-0.463
11,15,major > 10,1654,2.275,-0.649,-0.720,-0.569
14,30,on time < 1,104633,0.767,-0.106,-0.113,-0.101
17,30,minor 1-3,61680,0.857,-0.193,-0.201,-0.186
20,30,moderate 3-10,14739,1.270,-0.932,-0.963,-0.899
23,30,major > 10,1654,3.238,-1.256,-1.411,-1.083
26,60,on time < 1,104633,0.909,-0.105,-0.112,-0.098
29,60,minor 1-3,61680,1.009,-0.208,-0.217,-0.200


In [11]:
# 3.3 Chart: MAE by current delay, historical vs XGBoost, at each horizon
r = by_d0[by_d0["model"].isin(["historical", "xgb_full"])]
fig = px.bar(r, x="d0_bucket", y="mae", color="model", barmode="group", facet_col="horizon_min",
             color_discrete_map={"historical": SERIES[1], "xgb_full": SERIES[0]},
             title="Error by current delay: historical baseline vs XGBoost")
fig.update_layout(yaxis_title="MAE (min)", legend_title=None)
fig.update_xaxes(title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("horizon_min=", "") + " min"))
fig.show()

> **Finding 3.2–3.3 (by current delay).** The gain is concentrated where it matters:
>
> | Current delay | n (15 min) | Historical → XGBoost, 15 / 30 / 60 min | Gain |
> |---|---|---|---|
> | on time (< 1 min) | 104,633 | 0.78 → 0.67 / 0.87 → 0.77 / 1.01 → 0.91 | −14 / −12 / −10 % |
> | minor (1–3) | 61,680 | 0.87 → 0.72 / 1.05 → 0.86 / 1.22 → 1.01 | −18 / −18 / −17 % |
> | moderate (3–10) | 14,739 | 1.40 → 0.91 / 2.20 → 1.27 / 2.87 → 1.49 | **−35 / −42 / −48 %** |
> | major (> 10) | 1,654 | 2.92 → 2.28 / 4.49 → 3.24 / 6.27 → 4.31 | −22 / −28 / −31 % |
>
> - For late trains the historical baseline ignores the current delay; XGBoost learns how much of it will be recovered, which is worth up to **−1.4 min at 60 min** for moderately late trains and −2.0 min for major delays.
> - Ridge cannot do this: for major delays it is worse than the historical baseline.
> - Errors for trains > 10 min late remain large (2.3 / 3.2 / 4.3 min): the hard cases are still hard.

In [12]:
# 3.4 Normal vs disruption days (validation has one disruption day: indicative only)
by_dis = evaluate(common, ["pred_historical", "pred_xgb_full"], group_cols=("horizon_min", "is_disruption_day"),
                  reference="pred_historical", n_boot=300)
by_dis.pivot_table(index="is_disruption_day", columns=["horizon_min", "model"], values="mae").round(3)

horizon_min               15                  30                  60         
model             historical xgb_full historical xgb_full historical xgb_full
is_disruption_day                                                            
False                  0.878    0.717      1.066    0.854      1.269    1.009
True                   1.009    0.831      1.275    1.043      1.601    1.340

> **Finding 3.4.** On the single validation disruption day (21 May 2026), every model is worse, and XGBoost keeps a similar relative gain: 0.83 / 1.04 / 1.34 min vs 1.01 / 1.28 / 1.60 for the historical baseline (−18 / −18 / −16 %). Indicative only (one day); the normal-vs-disruption comparison is for the test and production months.

## 4. What the champion model uses

Gain importance of the saved XGBoost models (`models/xgb_full_h*.json`): the total loss reduction brought by the splits on each feature, as a share of the model's total.

In [13]:
# 4.1 Gain importance by feature and by feature group
group_of = {f: g for g, fs in FEATURE_GROUPS.items() for f in fs}
imp = []
for h in config.HORIZONS_MIN:
    booster = xgb.Booster()
    booster.load_model(config.ROOT / "models" / f"xgb_full_h{h}.json")
    gain = booster.get_score(importance_type="total_gain")
    total = sum(gain.values())
    imp += [{"horizon_min": h, "feature": f, "group": group_of.get(f, "?"), "share": g / total}
            for f, g in gain.items()]
imp = pd.DataFrame(imp)
display(imp.pivot_table(index="group", columns="horizon_min", values="share", aggfunc="sum")
          .sort_values(15, ascending=False).mul(100).round(1))
imp.pivot_table(index="feature", columns="horizon_min", values="share").sort_values(15, ascending=False).head(15).mul(100).round(1)

horizon_min,15,30,60
group,,,
prior,44.400,37.800,32.100
slack,19.500,13.100,8.100
current,16.200,29.800,45.000
timetable,10.300,8.400,6.400
context,4.500,4.800,4.200
history_1,2.000,2.300,2.000
history_5,1.700,2.100,1.200
history_3,1.500,1.600,1.100


horizon_min,15,30,60
feature,,,
pred_historical,44.400,37.800,32.100
d0_min,13.000,25.300,39.600
slack_to_target_min,12.200,6.600,3.700
run_reserve_to_target_min,4.800,4.100,2.800
dwell_reserve_to_target_min,2.400,2.400,1.600
sched_gap_min,2.300,1.700,1.200
operator_abbr,2.200,1.900,1.900
n_stops_to_target,2.100,1.100,0.900
arr_delay_now_min,1.900,3.100,4.200


In [14]:
# 4.2 Chart: top 12 features at 15 min
top = imp[imp["horizon_min"] == 15].nlargest(12, "share").sort_values("share")
fig = px.bar(top, x="share", y="feature", color="group", orientation="h",
             title="XGBoost (15 min): share of total gain by feature")
fig.update_layout(xaxis_tickformat=".0%", xaxis_title=None, yaxis_title=None, legend_title=None)
fig.show()

> **Finding 4.1–4.2 (gain importance).**
> - **The model switches strategy with the horizon:** at 15 min it relies on the historical prior (44 %) and slack (20 %), i.e. on the structure of the timetable; at 60 min the **current delay** dominates (`d0_min` alone 40 %, current group 45 %), i.e. on how much of today's delay will be recovered.
> - Slack matters most at short horizons (20 / 13 / 8 %), history features weigh ~5 % in total, context ~5 %.
> - Gain importance and ablations tell different stories for the prior: high gain (32–44 %) but small loss when removed (Finding 2.1). The information it carries (where and when) is largely redundant with the timetable and context features, so the trees can do without it. Ablations, not gain, measure what a feature group adds.

## 5. Findings and decisions

### Results (validation, common subset, MAE in min at 15 / 30 / 60 min)

| Model | MAE | vs historical |
|---|---|---|
| Persistence | 1.33 / 1.47 / 1.64 | |
| Constant offset | 1.06 / 1.25 / 1.44 | |
| Historical median | 0.88 / 1.07 / 1.28 | reference |
| Ridge (`full`) | 0.87 / 1.03 / 1.20 | −1 / −4 / −6 % |
| **XGBoost (`full`, refit)** | **0.72 / 0.86 / 1.02** | **−18 / −20 / −20 %** |

### Answers to the research questions so far

- **Does the train's recent trajectory help beyond its current delay?** No. Once the current departure delay, arrival delay and dwell are known, 1 to 5 previous stops add nothing (±0.003 min).
- **Does timetable slack explain recovery?** Partly: it is the only group that adds information beyond the current state, −0.011 to −0.015 min at every horizon, and it matters most at short horizons.
- **Is a linear model enough?** No. With identical features, XGBoost beats Ridge by 0.15–0.18 min. Recovery depends non-linearly on the current delay (gains of 35–48 % for trains 3–10 min late).
- **Where do the models gain?** Mostly on late trains, and on EC (the hardest category). On-time trains gain 10–14 %.

### Decisions

| # | Topic | Decision | Evidence |
|---|---|---|---|
| 1 | Champion | **XGBoost, `full` feature set, refit on Aug–Apr**, absolute-error objective, learning rate 0.1, one model per horizon | 1.1–1.2, 2.1 |
| 2 | Refit | Kept: −0.008 min at every horizon, significant | 2.2 |
| 3 | History features | Kept in `full` for comparability with the sequence model, although they add nothing | 2.1–2.3 |
| 4 | Learning rate | 0.1 (0.05 hit the 6,000-tree cap with the same accuracy and twice the time) | fit logs |
| 5 | Bar for the next models | XGBoost `full`: 0.720 / 0.860 / 1.020 min on validation. The network-feature gate compares XGBoost + network features against this model | 1.2 |
| 6 | Test month | Still untouched for the tabular models; evaluated once at the final evaluation | — |